# 3D Gaussian Splatting — Colab 학습 노트북

> **런타임 설정**: 런타임 → 런타임 유형 변경 → **GPU → A100**
>
> CUDA 11.8 설치 셀 실행 후 **반드시 런타임 재시작**이 필요합니다.

## 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 경로 설정
> ✏️ **여기서 경로만 수정하면 이후 모든 셀에 자동 반영됩니다.**

In [ ]:
import os, datetime

# ✏️ Drive 내 데이터 폴더 경로 (COLMAP 전처리 완료된 폴더)
DATA_PATH = '/content/drive/MyDrive/3dgs/data'

# ✏️ 실험 이름 태그
EXP_TAG = 'exp'

# 타임스탬프 + 태그로 출력 경로 자동 생성
timestamp   = datetime.datetime.now().strftime('%Y-%m-%d_%H:%M:%S')
OUTPUT_PATH = f'/content/drive/MyDrive/3dgs/output/{timestamp}_{EXP_TAG}'

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'DATA_PATH  : {DATA_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')

## 2. 3DGS 레포 클론
`--recursive`로 submodule까지 함께 클론합니다.

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git

## 3. 빌드 도구 및 기본 패키지 설치

In [ ]:
!apt-get update -qq
!apt-get install -y build-essential cmake ninja-build git wget
!pip install -q plyfile

## 4. COLMAP 설치

In [ ]:
!apt-get install -y colmap
!colmap --version

## 5. CUDA 11.8 설치
> ⚠️ **이 셀 실행 완료 후 런타임을 재시작하세요.**
> 메뉴: 런타임 → 런타임 다시 시작
> 재시작 후 **6번 셀(환경변수 설정)부터** 실행하세요.

In [ ]:
import subprocess, os
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

if 'release 12' in result.stdout or 'release 11' not in result.stdout:
    print('Installing CUDA 11.8...')
    os.system('wget -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run')
    os.system('sh cuda_11.8.0_520.61.05_linux.run --silent --toolkit')
    print('Done. ⚠️  Runtime restart required!')
else:
    print('CUDA 11.x already present, skipping.')

## 6. CUDA 11.8 환경 변수 설정
**런타임 재시작 후 이 셀부터 실행합니다.**

In [ ]:
import os, datetime

os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] = f"/usr/local/cuda-11.8/bin:{os.environ['PATH']}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-11.8/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

# 런타임 재시작으로 변수 초기화 → 재선언
# ✏️ 셀 1과 동일하게 맞춰주세요
DATA_PATH = '/content/drive/MyDrive/3dgs/data'
EXP_TAG   = 'exp'

timestamp   = datetime.datetime.now().strftime('%Y-%m-%d_%H:%M:%S')
OUTPUT_PATH = f'/content/drive/MyDrive/3dgs/output/{timestamp}_{EXP_TAG}'

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'DATA_PATH  : {DATA_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')

!nvcc --version

## 7. PyTorch 재설치 (CUDA 11.8 호환)

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 \
    --index-url https://download.pytorch.org/whl/cu118
!pip install numpy==1.26.4

## 8. torch.py 충돌 확인

In [ ]:
import os
if os.path.exists('./torch.py'):
    os.rename('torch.py', 'torch_local.py')
    print('Renamed torch.py -> torch_local.py')
else:
    print('✅ No conflict.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

## 9. Submodule 빌드 및 설치
`diff-gaussian-rasterization`과 `simple-knn`을 빌드합니다. (각 5~10분 소요)

In [ ]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!python setup.py install

In [ ]:
%cd /content/gaussian-splatting/submodules/simple-knn
!python setup.py install

## 10. 설치 확인

In [ ]:
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print('✅ diff_gaussian_rasterization OK')
except ImportError as e:
    print(f'❌ {e}')

try:
    import simple_knn
    print('✅ simple_knn OK')
except ImportError as e:
    print(f'❌ {e}')

## 11. 학습 → 렌더링 → 평가 (Train → Render → Metrics)

파라미터를 수정한 뒤 셀을 실행하면 **학습 → 렌더링 → 평가가 순서대로 자동 실행**됩니다.

```
OUTPUT_PATH/
└── point_cloud/
    ├── iteration_10000/point_cloud.ply
    ├── iteration_20000/point_cloud.ply
    └── iteration_30000/point_cloud.ply
```

In [ ]:
import os, textwrap
from pathlib import Path

# ── 파라미터 ─────────────────────────────────────────────
ITERATIONS      = 30000
SAVE_ITERATIONS = '10000 20000 30000'
# ─────────────────────────────────────────────────────────

%cd /content/gaussian-splatting

script = textwrap.dedent(f"""
    #!/bin/bash
    set -e
    ulimit -n 4096

    echo '===== [1/3] Training ====='
    python train.py \\
        -s {DATA_PATH} \\
        -m {OUTPUT_PATH} \\
        --iterations {ITERATIONS} \\
        --save_iterations {SAVE_ITERATIONS}

    echo '===== [2/3] Rendering ====='
    python render.py -m {OUTPUT_PATH}

    echo '===== [3/3] Metrics ====='
    python metrics.py -m {OUTPUT_PATH}
""").strip()

script_path = Path('scripts/run_3dgs.sh')
script_path.parent.mkdir(exist_ok=True)
script_path.write_text(script, encoding='utf-8')

print(script_path.read_text())
print('\n' + '='*50)
print('Starting pipeline...')
print('='*50 + '\n')

os.system(f'bash {script_path}')

## 12. 결과 확인

In [ ]:
import os

print('=== OUTPUT_PATH 구조 ===')
for root, dirs, files in os.walk(OUTPUT_PATH):
    depth = root.replace(OUTPUT_PATH, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 3:
        for f in files:
            print(f'{indent}  {f}')